# Защита персональных данных клиентов

<a id='check1'></a>
## Ключевые выводы

**Задача:** Разработать метод шифрования персональных данных клиентов страховой компании так, чтобы по зашифрованным данным было невозможно восстановить исходные, а качество моделей машинного обучения (линейной регрессии) оставалось неизменным.

**Решение:**  
Доказано математически и проверено экспериментально: умножение матрицы признаков на любую **обратимую матрицу** не меняет предсказания и метрики качества линейной регрессии.  
На основе этого факта предложен алгоритм:  
1. Генерируется случайная квадратная матрица с ненулевым определителем.  
2. Обучающая и тестовая выборки умножаются на эту матрицу.  
3. Зашифрованные данные передаются в отдел моделирования, где строится регрессия с точно такими же результатами.

**Проверка:**  
Коэффициент детерминации **R² = 0.40** на исходных данных и **R² = 0.40** на зашифрованных (различие в 15-м знаке после запятой, вызванное погрешностью вычислений). Параметры модели изменились, но её предсказательная способность осталась прежней.

**Преимущества метода:**  
- Не требует передачи ключа шифрования в точку обучения модели.  
- Может использоваться с любой обратимой матрицей, в том числе генерируемой динамически.  
- Гарантирует безопасность персональных данных без потери качества прогнозов.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from pathlib import Path

rand = 484848

## Содержание

- [Ключевые выводы](#check1)
- [Вступление](#check2)
- [Загрузка и подготовка данных](#check3)        
- [Умножение матриц](#check4)
- [Алгоритм преобразования](#check5)
- [Проверка алгоритма](#check)
- [Итоговые выводы](#check6)

<a id='check2'></a>
## Вступление

Нашей задачей является предложение метода шифрования данных клиентов для страховой компании так, чтобы шифрование не меняло качества регрессии, нужно для построения прогнозов из этих данных. Кроме неизменности качества машинного обучения других требований к шифрованию не ставится.

Обсудим план наших действий.

В работе мы: 
1. изучим и проверим предоставленные данные
2. Ответим на ни на что не намекающий вопрос - изменится ли качество линейной регрессии при умножении матрицы признаков на обратимую матрицу? Докажем, что нет.
3. Предложим на основании этого алгоритм преобразования данных
4. Проверим его руками и убедимся, что регрессия не теряет в качестве

Вам нужно защитить данные клиентов страховой компании «Хоть потоп». Разработайте такой метод преобразования данных, чтобы по ним было сложно восстановить персональную информацию. Обоснуйте корректность его работы.

Нужно защитить данные, чтобы при преобразовании качество моделей машинного обучения не ухудшилось. Подбирать наилучшую модель не требуется.

<a id='check3'></a>
## Загрузка и подготовка данных

Изучим предоставленные данные:

In [2]:
pd.set_option('display.max_columns', 50) 
# читаем файл
data_path = Path.cwd().parent / 'data' / 'insurance.csv'
raw_data = pd.read_csv(data_path, sep = ',')
# выводим обзорные данные о данных
raw_data.info()
display(raw_data.head(10))
display(raw_data.describe())
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Пол                5000 non-null   int64  
 1   Возраст            5000 non-null   float64
 2   Зарплата           5000 non-null   float64
 3   Члены семьи        5000 non-null   int64  
 4   Страховые выплаты  5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


,Пол,Возраст,Зарплата,Члены семьи,Страховые выплаты
0,1,41.0,49600.0,1,0
1,0,46.0,38000.0,1,1
2,0,29.0,21000.0,0,0
3,0,21.0,41700.0,2,0
4,1,28.0,26100.0,0,0
5,1,43.0,41000.0,2,1
6,1,39.0,39700.0,2,0
7,1,25.0,38600.0,4,0
8,1,36.0,49700.0,1,0
9,1,32.0,51700.0,1,0


,Пол,Возраст,Зарплата,Члены семьи,Страховые выплаты
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,0.499000,30.952800,39916.360000,1.194200,0.148000
std,0.500049,8.440807,9900.083569,1.091387,0.463183
min,0.000000,18.000000,5300.000000,0.000000,0.000000
25%,0.000000,24.000000,33300.000000,0.000000,0.000000
50%,0.000000,30.000000,40200.000000,1.000000,0.000000
75%,1.000000,37.000000,46600.000000,2.000000,0.000000
max,1.000000,65.000000,79000.000000,6.000000,5.000000


Битых данных нет, есть соблазн перевести вещественные данные в целочисленные, но далее мы все равно их будем умножать на вещественную случайную матрицу, а потом восстанавливать данные, получая вещественные, так что это лишено смысла.

Визуальный анализ нам не нужен в рамках задачи. Проверка на пропуски на всякий случай:

In [3]:
for column_name in raw_data.columns:
    print(f'Пропусков в {column_name}:', raw_data[column_name].isna().sum())
# print чтобы переносов не было

Пропусков в Пол: 0
Пропусков в Возраст: 0
Пропусков в Зарплата: 0
Пропусков в Члены семьи: 0
Пропусков в Страховые выплаты: 0


Все чисто, можно переходить к высчислениям.

**Краткий итог:** данные без ошибок и пропусков, преобразовывать их дополнительно сочтено не нужным.

<a id='check4'></a>
## Умножение матриц

Обозначения:

- $X$ — матрица признаков (нулевой столбец состоит из единиц)

- $y$ — вектор целевого признака

- $P$ — матрица, на которую умножаются признаки

- $w$ — вектор весов линейной регрессии (нулевой элемент равен сдвигу)

**Вопрос:** изменится ли качество линейной регрессии при умножении матрицы признаков на обратимую матрицу?

**Ответ:** Ничего не изменится, так как такое умножение приводит к тем же значениям вектора предсказаний для заданного ветора целевого признака. Это задача на линейную алгебру. Проверим это нехитрый факт.

**Обоснование**

Наше утверждение состоит в том, что формула обучения:
    
$$
w = (X^T X)^{-1} X^T y
$$

даст тот же вектор предсказания, что и формула обучения 

$$
v = ((XP)^T (XP))^{-1} (XP)^T y
$$

где $P$ — обратимая матрица, на которую могут быть умножены признаки X

Нам потребуется незаурядное умение раскрывать скобочки и свойства обратимых матриц, состоящее в том, что для любых обратимых матриц A и B верно

$$
(AB)^{-1} = B^{-1} A^{-1}
$$

и

$$
(AB)^T = B^T A^T
$$

и, наконец,
$$
(A^T)^{-1} = (A^{-1})^T
$$
"Интресные утверждения. Проверять я их, конечно же, не буду." Так как там алгебраческие дополнения, детерминанты и прочие ужасы с первого курса университета.

Другим не очевидным фактом является ассоциативность умножения матриц - т.е. возможность в умножении раскрывать скобки как хочется. Доказательство этого факта также входит в обычный университетский курс линейной алгебры. Мы его не будем расписывать, хотя оно достаточно просто.

Раскрываем скобочки в
$$
((XP)^T (XP))^{-1} (XP)^T
$$

Вперед:

$$
((XP)^T (XP))^{-1}(XP)^T = (P^TX^TXP)^{-1}(XP)^T=(P^T(X^TX)P)^{-1}(XP)^T
$$
обратим внимание что $$X^TX$$ - обратимая квадратная матрица (см. выше, а произведение обратимых квадратных матриц очевидно обратимо, соотв.

$$
(P^T(X^TX)P)^{-1}(XP)^T= (P^T((X^TX)P))^{-1}(XP)^T=(X^TX)P)^{-1}(P^T)^{-1}(XP)^T=
$$
$$
=P^{-1}(X^TX)^{-1}(P^T)^{-1}(XP)^T=P^{-1}(X^TX)^{-1}(P^T)^{-1}P^TX^T=P^{-1}(X^TX)^{-1}EX^T=P^{-1}(X^TX)^{-1}X^T
$$

Хорошо, теперь, чтобы получить вектор предсказания, нужно взять
$$
b = XPv = XP((XP)^T (XP))^{-1} (XP)^Ty =  XPP^{-1}(X^TX)^{-1}X^Ty= XE(X^TX)^{-1}X^Ty = X(X^TX)^{-1}X^Ty= Xw = a
$$

Что и требовалось доказать.

Из последней строчки также видно, что для нового "зашифрованного" вектора решения:

$$
v = ((XP)^T (XP))^{-1} (XP)^T y= P^{-1}(X^TX)^{-1}X^T y = P^{-1}w
$$

Т.е. чтобы найти новый мин. вектор нужно просто умножить старый на обратную к P матрицу.

**Краткий итог:** умножение на обратимую матрицу не меняет качество линейной регрессии. Доказательство с использованием лемм приведено в теле раздела.

<a id='check5'></a>
## Алгоритм преобразования

**Алгоритм**

Естественно, после такой подготовительной работы уровня первого курса, мы будем умножать матрицу данных клиентов на обратимую *квадратную* матрицу справа. Естественно, ее будем генерировать случайным образом. Сторона квадратной матрицы должна быть равна числу признаков, очевидно, чтобы умножение могло быть возможно. По идее, можно было бы еще предложить какую-то биекцию для элементов матрицы данных клиентов, вроде линейного преобразования, но либо особо не защитит дополнительно, либо создаст большую нагрузку на сервера.Оставим просто матрицу.

Для проверки обратимости посчитаем детерминант случайной матрицы (для обратимости квадратной матрицы необходимо и достаточно чтобы детерминант был ненулевой). Скорее всего с первой итерации мы получим обратимую, мн-во необратимых матриц имеет меру 0 в пространстве всех матриц. Итого, алгоритм:

<ol>
<li>Генерируем случайную матрицу.</li>
<li>Проверяем ее детерминант. Если он нулевой, возвращаемся к шагу 1, если нет, идем дальше к шагу 3.</li>
<li>Умножаем матрицу признаков на полученную матрицу справа, получаем новую, конфиднциальную матрицу признаков.</li>
<li>Применяем линейную регрессию для преобразованных признаков.</li>    
</ol>

**Обоснование**

Умножение на квадратную обратимую матрицу справа никак не повлияет ни на количество признаков, ни на качество регрессии из-за того, что предсказания будут получены одни и те же в обоих случаях. Доказательство приведено выше в аункте 2. оличество признаков регрессии при умножении на квадратную матрицу тоже не поменяется.

Но мы также и проверим это "руками" в пункте 4 ниже.

**Краткий итог:** предсказуемым образом на основании части 3 мы предлагаем шифрование через случайную обратимую матрицу. Доказательство того, что это е повлияет на качество регрессии как раз преведено в 3 части.

<a id='check'></a>
## Проверка алгоритма

Проделаем стандартные действия по проработке модели линейной регрессии на данных. Разбиваем данные на тренировочные и валидационные:

In [4]:
# формируем признаки и цель регрессии
features = raw_data.drop('Страховые выплаты', axis=1)
target = raw_data['Страховые выплаты']
# разбиваем на выборки
features_train, features_valid, target_train, target_valid = \
train_test_split(features, target, test_size=0.25, random_state=rand)

Теперь проверим регрессию на исходных данных:

In [5]:
def check_reg(features_train, target_train, features_valid, target_valid):
    model = LinearRegression()
    model.fit(features_train, target_train)
    R2_orig = r2_score(target_valid, model.predict(features_valid))
    display('Получившиеся коэффициенты регрессии:', np.ndarray.tolist(model.coef_))
    display('Получившийся R2', R2_orig)
    
check_reg(features_train, target_train, features_valid, target_valid)

'Получившиеся коэффициенты регрессии:'

[0.012199115329431057,
 0.036573727712659064,
 -4.3668964144546214e-07,
 -0.013371755106949119]

'Получившийся R2'

0.40393792634767445

Так себе регрессия и R-2 так себе. Проверим что не сделаем хуже при помощи случайной матрицы. Сначала генерируем матрицу (у нас четыре члена регрессии):

In [6]:
 def get_rand_matrix(data):
        det = 0
        while det == 0:
            matrix = np.random.normal(size=(data.shape[1], data.shape[1]))
            det = np.linalg.det(matrix)
        return matrix


encryptor = get_rand_matrix(features)

display('Получившаяся матрица:')
display(encryptor)
# Смогтрим на обратимость матрицы:
display('Ее детерминант:')
display(np.linalg.det(encryptor))
# Даже посчитаем обратную матрицу:
display('Обратная к ней матрица:')
display(np.linalg.inv(encryptor))

'Получившаяся матрица:'

array([[-0.67923915, -0.70980207, -1.10042323,  2.04363853],
       [-0.68624182, -0.45359331,  0.02079625, -0.70337066],
       [-1.07794139,  0.20888612, -1.83374861, -0.19859838],
       [-0.5991696 , -0.88673068, -1.04462829,  0.42205369]])

'Ее детерминант:'

-2.3127735038218966

'Обратная к ней матрица:'

array([[-0.75414958, -1.37028195, -0.26995337,  1.24102862],
       [ 0.28805139, -0.03928689,  0.51934291, -1.21587728],
       [ 0.41522972,  0.80488849, -0.31871541, -0.81918692],
       [ 0.562301  , -0.03568002, -0.08096053, -0.45092578]])

Хорошо, преобразуем данные матричным умножением на новую матрицу:

In [7]:
features_enc = features @ encryptor
display(features_enc.head(10))

,0,1,2,3
0,-53495.307423,10340.557557,-90955.223271,-9876.852287
1,-40993.939233,7915.920433,-69682.535035,-7578.671540
2,-22656.670269,4373.454256,-38508.117638,-4190.963786
3,-44965.765511,8699.252169,-76468.969412,-8295.479235
4,-28154.164371,5438.517246,-47861.356749,-5201.068529
5,-44226.983095,8542.343032,-75185.988294,-8169.890883
6,-42822.914317,8272.605453,-72802.198291,-7908.899503
7,-41628.769738,8047.407569,-70787.455228,-7679.749986
8,-53599.670353,10363.714136,-91138.702113,-9893.195272
9,-55752.808172,10783.300744,-94806.282510,-10287.578555


Получив новые, 100% защищенные на уровне примерно 100% российских компаний, считаем еще раз регрессию:

In [8]:
# разбиваем на выборки новые данные:
features_train_enc, features_valid_enc, target_train_enc, target_valid_enc = \
train_test_split(features_enc, target, test_size=0.25, random_state=rand)

display('Параметры старой регрессии:')
check_reg(features_train, target_train, features_valid, target_valid)
display('Параметры закодированной регрессии:')
check_reg(features_train_enc, target_train_enc, features_valid_enc, target_valid_enc)

'Параметры старой регрессии:'

'Получившиеся коэффициенты регрессии:'

[0.012199115329431057,
 0.036573727712659064,
 -4.3668964144546214e-07,
 -0.013371755106949119]

'Получившийся R2'

0.40393792634767445

'Параметры закодированной регрессии:'

'Получившиеся коэффициенты регрессии:'

[-0.0759108896556048,
 0.01833529057674557,
 0.04545731375744459,
 0.011584327739602282]

'Получившийся R2'

0.4039379263474341

R2 совпадает, качество не изменилось. Разница в R2 на конце - это заговор питона против нас, так как он не умеет хранить нормально такие длинные числа. Вектор же коэффициентов регрессии же поменялся, что логично. Таким образом, мы убедили даже неверующих в математику, что наш метод работает. Воистинну, презентация на уровне службы безопасности российских банков.

**Краткий итог:** на примере линейной регрессии мы проверили параметры ее для данных без шифрования, затем зашифровали их через сгенерированную случайную матрицу, и сравнили R2 для обоих случаев. Хотя коэффициенты регресии предсказуемо отличаются, R2 остался неизменным. НГаш метод работает.

<a id='check6'></a>
## Итоговые выводы

В работе мы предложили способ шифрования данных клиентов, не влияющий на качество машинного обучения. <br>

Работа состояла из следующих шагов:
<ol>
<li>Обзор данных</li>
<li>Теорема о том, что при умножении на любую обратимую матрицу качество линейной регрессии не изменится.</li>
<li>Предложение и обсуждение алгоритма шифрования на основании обратимой матрицы.</li>
<li>Практическое применение алгоритма шифрования и сравнение качества регрессии до и после</li>    
</ol>

Последний пункт показал, что:

In [9]:
display('Параметры старой регрессии:')
check_reg(features_train, target_train, features_valid, target_valid)
display('Параметры закодированной регрессии:')
check_reg(features_train_enc, target_train_enc, features_valid_enc, target_valid_enc)

'Параметры старой регрессии:'

'Получившиеся коэффициенты регрессии:'

[0.012199115329431057,
 0.036573727712659064,
 -4.3668964144546214e-07,
 -0.013371755106949119]

'Получившийся R2'

0.40393792634767445

'Параметры закодированной регрессии:'

'Получившиеся коэффициенты регрессии:'

[-0.0759108896556048,
 0.01833529057674557,
 0.04545731375744459,
 0.011584327739602282]

'Получившийся R2'

0.4039379263474341

Параметры регрессии действительно не меняются и мы можем генерировать случайную матрицу и умножать ее на нее данные без потерь в качестве машинного обучения:

In [10]:
encryptor_show = get_rand_matrix(features)

display('Получившаяся матрица:')
display(encryptor_show)
# Смогтрим на обратимость матрицы:
display('Ее детерминант:')
display(np.linalg.det(encryptor_show))


'Получившаяся матрица:'

array([[-0.97419353,  0.79559133,  1.10546487,  0.75012346],
       [ 0.66882897, -0.37328421, -1.29176719, -0.93609038],
       [ 1.58820525, -0.71890273, -0.25317443, -0.29854504],
       [ 0.69554004, -1.23009773,  0.99278577, -1.78371756]])

'Ее детерминант:'

1.5034714486810796

Простое умножение данных на такую матрицу - допустимый способ шифрования:

In [11]:
features_enc = features @ encryptor_show
display(features_enc.head(10))

,0,1,2,3
0,78802.123499,-35673.314376,-12608.315766,-14847.247331
1,60383.260992,-27336.704765,-9679.056718,-11389.555431
2,33371.706190,-15107.782491,-5354.124208,-6296.592481
3,66243.595215,-29988.542844,-10582.515131,-12472.553541
4,41469.909919,-18773.017519,-6642.916552,-7817.485976
5,65145.591588,-29492.727597,-10432.606446,-12283.415877
6,63078.249453,-28556.660916,-10098.312622,-11891.562962
7,61323.251158,-27763.102134,-9799.750441,-11553.625587
8,78957.599878,-35743.338227,-12627.174373,-14872.421383
9,82131.335053,-37179.650543,-13128.356158,-15465.767103


Данные при этом уже не пересобрать на конце где будет проходить машинное обучение. Задачи работы выполнены.